#### **Initialization**

In [1]:
# CITE THE LIBRARYYYYYYYYYYYYYYYYYY
import openai
import os
import json

# Security Measure
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

openai.api_key  = os.getenv('OPENAI_API_KEY')

##### Load the KG:

In [2]:
import rdflib

eu_graph = rdflib.Dataset().parse("eu_legislation_graph.nq", format="nquads")

#### Prepare the Agent class:

In [3]:
class Legal_Agent:
    def __init__(self, system_prompt="", step_prompts=[], api_key="", tool_table=dict()):
        self.system_prompt = system_prompt
        self.step_prompts = step_prompts
        self.context = [{"role": "system", "content": system_prompt}]
        self.reasoning_log = []
        self.tool_lookup = {value['function']['name']: func for func, value in tool_table.items()}
        self.tools = [value for value in tool_table.values()]
        self.instance = openai.OpenAI(
            base_url = "https://integrate.api.nvidia.com/v1",
            api_key = api_key
        )

    def respond_to(self, query="", role="user"):
        # Only append if there's actual user content to add
        if query:
            self.context.append({"role": role, "content": query})

        answer = self.instance.chat.completions.create(
            model="openai/gpt-oss-120b",
            reasoning_effort="high",
            messages=self.context,
            temperature=0,
            top_p=0.1,
            tools=self.tools,
            max_tokens=None,
            stream=False
        )

        reasoning = getattr(answer.choices[0].message, "reasoning_content", None)
        self.reasoning_log.append(str(reasoning))

        if answer.choices[0].message.tool_calls:
            print("Calling Tool...")

            # ✅ Append the full assistant message (with tool_calls) to context
            self.context.append(answer.choices[0].message)

            tool_call = answer.choices[0].message.tool_calls[0]
            tool_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            tool_result = self.tool_lookup[tool_name](**args)

            print("Processing tool results...")

            # ✅ Include tool_call_id to match the call
            self.context.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": tool_name,
                "content": str(tool_result)
            })

            return self.respond_to()  # No role/query needed — context is already updated

        else:
            self.context.append({
                "role": "assistant",
                "content": answer.choices[0].message.content
            })
            return answer.choices[0].message.content

    def sequential_reasoning(self, query="", max_depth=10):
        state = {"query": query}
        return self.conditional_call(
            step_index=0,
            depth=0,
            max_depth=max_depth,
            state=state
            )

    def conditional_call(self, step_index, depth, max_depth, state):
        # print("Step", step_index + 1)     # For debugging
        if depth > max_depth:
            raise RuntimeError("Max reasoning depth reached")

        prompt = self.step_prompts[step_index].format(**state)
        raw_result = self.respond_to(prompt, "user")
        state = json.loads(raw_result)

        next_step = state.get("next_step", None)

        if next_step is None:
            return state

        return self.conditional_call(
            step_index=(next_step - 1),
            depth=(depth + 1),
            max_depth=max_depth,
            state=state
            )

    def reset_working_memory(self):
        self.context = [{"role": "system", "content": self.system_prompt}]
        self.reasoning_log = []

    def reset_context(self):
        self.context = [{"role": "system", "content": self.system_prompt}]    

    def export_context(self):
        return self.context

#### Prompts:

In [4]:
# Cite the SOURCEEEEEEEEEEEEEEE!!!!!!!!!!!!!!!!!!!!!!!!!
practice_areas = [
    "Bankruptcy and insolvency law",
    "Commercial law",
    "Consumer law",
    "Criminal law",
    "Employment law",
    "EU law",
    "Family law",
    "Human rights and civil liberties",
    "Immigration and asylum law",
    "Intellectual property",
    "Information Technology (IT) law",
    "Litigation, mediation, arbitration",
    "Personal injury, damage to goods",
    "Property law",
    "Public law",
    "Social security law",
    "Succession law",
    "Tax law",
    "Traffic and transport law"
]

# EQUIP THE RECEPTIONIST WITH A QUERY TOOL?
receptionist_prompt = f"""
You are a front-facing receptionist that handles inquiries about EU law.
You are required to do the following:

1.  Reformulate and improve queries;
2.  Identify ALL practice areas RELEVANT to the reformulated query;
3.  Respond to queries accordingly.

You will be prompted to do each of these tasks, one at a time.
"""

receptionist_step_prompts = [
    """
    1.  Reformulate the user's query via decomposition and/or adding sufficient context.
        Ensure that queries/sub-queries are atomic to facilitate downstream SPARQL queries.
        Collect the resulting query/sub-queries in a list.
        Determine if the user's query is general or complex.
        If the query is general, proceed to step 3 after the outputting the JSON dictionary.

        Output the following JSON dictionary:
        {{
            "improved_query": [reformulated_query_list],
            "next_step": 2 or 3 # depending on step 1
        }}

        User query: {query}
    """,

    """
    2.  List ALL practice areas from the list below RELEVANT to the reformulated query:
        Bankruptcy and insolvency law, Commercial law, Consumer law, Criminal law, Employment law, EU law, Family law, Human rights and civil liberties, Immigration and asylum law, Intellectual property, Information Technology (IT) law, Litigation, mediation, arbitration, Personal injury, damage to goods, Property law, Public law, Social security law, Succession law, Tax law, Traffic and transport law

        Output the following JSON dictionary:
        {{
            "improved_query": [reformulated_query_list],
            "practice_areas": ["area_1", "area_2", "area_3", ...],
            "next_step": 3
        }}

        Reformulated Query: {improved_query}
    """,

    """
    3.  Respond to the reformulated query.
        If you performed step 2, inform the user that you shall forward the query all relevant experts in the identified practice areas instead.

        Output the following JSON dictionary:
        {{
            "improved_query": [reformulated_query_list],
            "practice_areas": ["area_1", "area_2", "area_3", ...],
            "response": "your_response"
        }}

        Reformulated query: {improved_query}
    """
]

sparql_agent_prompt = """
You are a SPARQL sub-agent. You construct and execute SPARQL queries
using the sparql_query tool, then return structured results. You do NOT
answer the user directly or interpret results beyond verbalization.

────────────────────────────────────────
WORKFLOW (follow exactly, in order)
────────────────────────────────────────
1. Extract and expand keywords from the user's query (see KEYWORD RULES).
2. Construct ONE valid SPARQL SELECT query using those keywords.
3. Execute it with sparql_query(query).
4. Evaluate the returned URIs against the user's query (see EVALUATION RULES).
   - If results are semantically consistent: proceed to step 5.
   - If results are semantically inconsistent or empty: revise keywords
     and retry ONCE. If the retry fails or is still inconsistent,
     return the error or best available result.
5. Return the JSON output block below. Do nothing else.

────────────────────────────────────────
GRAPH CONTENT RULES
────────────────────────────────────────
This graph contains NO literals. Every value — entities, types,
properties, and provenance — is a URI.

You do NOT know the namespaces, the exact local names, or the full
URIs of any resource in this graph. Do NOT assume, invent, or guess
any namespace or URI. Do NOT declare PREFIX blocks. Do NOT use
prefixed terms for anything other than rdf:type, which is a fixed
W3C built-in.

────────────────────────────────────────
KEYWORD RULES
────────────────────────────────────────
Before constructing any query, derive and expand keywords for each
resource role (subject type, predicate, object type) directly from
the user's natural language query.

Step 1 — Extract:
  Use the most specific content word the user provided for each role.
  Do NOT substitute synonyms or ontology jargon not present in the query.
  If a role is ambiguous, leave that variable unconstrained.

Step 2 — Stem:
  Strip inflections to get a root form that matches URI variants broadly.
  Examples:
    "regulations"  → "regulat"
    "citations"    → "citat"
    "directives"   → "direct"
    "judgments"    → "judg"
    "references"   → "refer"

Step 3 — Expand semantically:
  For each stemmed keyword, generate 2-3 semantically related root
  variants that a URI designer might plausibly have used for the same
  concept. Combine them into a single regex alternation pattern.
  Examples:
    "regulat"  → "regulat|legislat|act"
    "citat"    → "citat|refer|mention"
    "judg"     → "judg|decision|ruling"
    "direct"   → "direct|legislat|measure"

  Rules for expansion:
  - Only include variants that are genuinely synonymous or
    conventionally interchangeable for the concept.
  - Do NOT expand into unrelated terms.
  - Prefer shorter roots to maximise URI coverage.

If a prior tool result in this session returned a URI, you may use
that exact URI directly in subsequent queries — copy it verbatim.

────────────────────────────────────────
URI MATCHING STRATEGY
────────────────────────────────────────
Use FILTER(regex(str(?x), ...)) as your ONLY matching strategy for
all resources — classes, predicates, and entities alike.

Rules:
- Always apply str() when matching a URI variable.
- Always use the case-insensitive flag "i".
- Use the expanded alternation pattern from Step 3 of KEYWORD RULES.
- Never anchor to a namespace IRI you have not seen in a tool result.
- Never use .* as a prefix or write patterns with no keyword content.

────────────────────────────────────────
EVALUATION RULES
────────────────────────────────────────
After receiving tool results, inspect the returned URIs and assess
whether they are semantically consistent with the user's query before
finalizing the response.

For each returned URI, ask:
- Does the local name of this URI relate to the concept the user asked about?
- Is this URI playing the correct role (subject, predicate, object)
  given the query structure?
- Are there URIs in the results that are clearly unrelated to the query
  (false positives introduced by an overly broad keyword)?

If results are consistent: proceed to output.

If results contain false positives: narrow the keyword pattern and retry.
  Example: "act" matched too broadly → tighten to "legal_act|enact"

If results are empty or clearly wrong: broaden or reframe the keyword
  pattern and retry.
  Example: "regulat" returned nothing → expand to "regulat|legislat|norm"

Only retry ONCE. On the second failure, return the best available
result with an explanation in the "error" field.

────────────────────────────────────────
QUERY RULES
────────────────────────────────────────
- SELECT only — never ASK, CONSTRUCT, or DESCRIBE.
- Always name variables explicitly — never SELECT *.
- No PREFIX declarations.
- Use GRAPH ?g { ... } whenever named graph provenance is relevant.
- Use OPTIONAL { } for properties that may not exist on all subjects.
- Always include LIMIT (default 100) unless the query is an aggregate.
- Pass the query as a raw string — no markdown fences, no quotes wrapping it.

────────────────────────────────────────
OUTPUT (return this JSON block only)
────────────────────────────────────────
{
  "user_query": "the original question asked",
  "sparql_query": "the executed SPARQL query",
  "keywords": {
    "extracted": {"subject": "...", "predicate": "...", "object": "..."},
    "expanded":  {"subject": "...", "predicate": "...", "object": "..."}
  },
  "raw_results": [ ...exact tool output rows... ],
  "verbalized_results": [ ...one plain-English sentence per row... ],
  "provenance": [ ...graph IRIs if GRAPH ?g was used, else []... ],
  "error": null
}

On failure:
{
  "user_query": "...",
  "sparql_query": "...",
  "keywords": {
    "extracted": {"subject": "...", "predicate": "...", "object": "..."},
    "expanded":  {"subject": "...", "predicate": "...", "object": "..."}
  },
  "raw_results": [],
  "verbalized_results": [],
  "provenance": [],
  "error": "exact error message from tool"
}
"""

legal_expert_list = []

for area in practice_areas:
    prompt = f"""
    You are a EU legal researcher in the following practice area: {area}.
    You are required to analyze the given queries/sub-queries STRICTLY WITHIN your practice area.
    You will be tasked to do the following:

    1.  For each query/sub-query, call a SPARQL sub-agent via the call_sparql_subagents tool.
    2.  Use the IRAC (Issue-Rules-Application-Conclusion) framework to analyze the results.
    3.  Organize the results into a coherent argument.

    You will be prompted to do each of these tasks, one at a time.
    """

    step_prompts = [
        """
        1.  For each query/sub-query, call a SPARQL sub-agent via the call_sparql_subagents tool.
            ONLY SEARCH FOR INFORMATION WITHIN YOUR PRACTICE AREA.

            Output the following JSON dictionary:
            {{
                "compiled_results": [list of passed statements/propositions],
                "queries_to_recall": [keep this blank, it's for Step 2],
                "next_step": 2
            }}

            Query List: {query}
        """,

        """
        2.  Evaluate the results returned by the SPARQL sub-agent(s) using the IRAC (Issue-Rules-Application-Conclusion) framework.
            If the results are satisfactory, move on to step 3;
            Otherwise, if the results are inadequate or incomprehensive,
                reformulate the corresponding query/subquery and call a SPARQL sub-agent via the sparql_agent tool.

            Output the following JSON dictionary:
            {{
                "compiled_results": [list of passed statements/propositions],
                "queries_to_recall": [list of reformulated queries],
                "next_step": 2 or 3 # depending on the results
            }}

            Compiled Results: {compiled_results}
            Query(ies) to Recall: {queries_to_recall}
        """,

        """
        3.  Organize the results into a coherent argument, as a list of propositions.
            LIST OUT ALL PROPOSITIONS, including refuting ones.

            Output the following JSON dictionary:
            {{
                "practice_area": "your practice area",
                "supporting_propositions": [list of supporting statements/propositions],
                "refuting_propositions": [list of refuting statements/propositions]
            }}

            "Compiled Results: {compiled_results}"
        """
    ]

    legal_expert_list.append({"prompt": prompt, "step_prompts": step_prompts, "practice_area": area})

expert_api_keys = [openai.api_key for expert in legal_expert_list]
for index, api in enumerate(expert_api_keys):
    legal_expert_list[index]["api_key"] = api

evaluator_prompt = """
You are a judge with comprehensive knowledge of EU law, across all but not limited to the following practice areas:
Bankruptcy and insolvency law, Commercial law, Consumer law, Criminal law, Employment law, EU law, Family law, Human rights and civil liberties, Immigration and asylum law, Intellectual property, Information Technology (IT) law, Litigation, mediation, arbitration, Personal injury, damage to goods, Property law, Public law, Social security law, Succession law, Tax law, Traffic and transport law

You are required to do the following:
1.  Generate a COMPREHENSIVE EVALUATION on the compiled analyses through the IRAC (Issue-Rules-Application-Conclusion) framework.
2.  Compile supporting and refuting statements/propositions.
2.  Spot inconsistencies or conflicting rules and highlight them.
3.  Provide recommendations/strategies based on your analysis, and predict the most likely outcome(s), if applicable.

Output Format:
{{
    "evaluation": "a comprehensive evaluation",
    "supporting_props": [list of supporting statements/propositions],
    "refuting_props": [list of refuting statements/propositions],
    "inconsistencies": [list of conflicting/inconsistent rules],
    "recs_and strats": [list of recommendations and strategies],
    "likely_outcomes": [most likely outcomes, if applicable]
}}

Compiled Analyses: {query}
"""

#### Tools:

In [5]:
from concurrent.futures import ThreadPoolExecutor

def sparql_query(query):
    """Returns results from a SPARQL query on a given RDF graph."""
    # print("----- SPARQL INPUT -----")
    # print(repr(query))
    # print("------------------------")

    query = query.strip()

    if query.startswith("'") and query.endswith("'"):
        query = query[1:-1]

    if query.startswith('"') and query.endswith('"'):
        query = query[1:-1]

    result = eu_graph.query(query)

    return [row for row in result]

sparql_tool_dict = {
        sparql_query: {
            "type": "function",
            "function": {
                "name": "sparql_query",
                "description": "Returns results from a SPARQL query on a given RDF graph.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "query": {
                            "type": "string",
                            "description": "A SPARQL query to execute."
                        }
                    },
                    "required": ["query"]
                }
            }
        }
    }

def call_sparql_subagents(queries):
    """Receives a list of queries, and instantiates sparql sub-agents to handle each query using SPARQL."""
    results = list()
    
    def sub_agent_thread(query):
        sub_agent = Legal_Agent(
                system_prompt=sparql_agent_prompt,
                api_key=openai.api_key,
                tool_table=sparql_tool_dict
            )
        return sub_agent.respond_to(query)

    with ThreadPoolExecutor() as executor:
        results = list(executor.map(sub_agent_thread, queries))

    return results

sparql_batch_tool_dict = {
    call_sparql_subagents: {
        "type": "function",
        "function": {
            "name": "call_sparql_subagents",
            "description": "Receives a list of queries, and instantiates sparql sub-agents to handle each query using SPARQL.",
            "parameters": {
                "type": "object",
                "properties": {
                    "queries": {
                        "type": "array",
                        "items": {
                            "type": "string"
                            },
                        "description": "A list of natural language queries to be converted into SPARQL."
                    }
                },
                "required": ["queries"]
            }
        }
    }
}

In [6]:
def describe_graph():
    """Returns classes, properties, and named graphs present in the RDF graph."""
    classes_query = """
        SELECT DISTINCT ?type (COUNT(?s) AS ?n)
        WHERE { ?s rdf:type ?type }
        GROUP BY ?type ORDER BY DESC(?n) LIMIT 30
    """
    props_query = """
        SELECT DISTINCT ?prop (COUNT(*) AS ?n)
        WHERE { ?s ?prop ?o }
        GROUP BY ?prop ORDER BY DESC(?n) LIMIT 50
    """
    graphs_query = "SELECT DISTINCT ?g WHERE { GRAPH ?g { ?s ?p ?o } }"
    return {
        "classes": [row for row in eu_graph.query(classes_query)],
        "properties": [row for row in eu_graph.query(props_query)],
        "named_graphs": [row for row in eu_graph.query(graphs_query)]
    }

#### *SPARQL Sub-Agent Testing:*

In [7]:
sparql_agent = Legal_Agent(
    system_prompt=sparql_agent_prompt,
    api_key=openai.api_key,
    tool_table=sparql_tool_dict
    )

In [8]:
sparql_agent.reset_context()

In [9]:
sparql_agent.context

[{'role': 'system',
  'content': '\nYou are a SPARQL sub-agent. You construct and execute SPARQL queries\nusing the sparql_query tool, then return structured results. You do NOT\nanswer the user directly or interpret results beyond verbalization.\n\n────────────────────────────────────────\nWORKFLOW (follow exactly, in order)\n────────────────────────────────────────\n1. Extract and expand keywords from the user\'s query (see KEYWORD RULES).\n2. Construct ONE valid SPARQL SELECT query using those keywords.\n3. Execute it with sparql_query(query).\n4. Evaluate the returned URIs against the user\'s query (see EVALUATION RULES).\n   - If results are semantically consistent: proceed to step 5.\n   - If results are semantically inconsistent or empty: revise keywords\n     and retry ONCE. If the retry fails or is still inconsistent,\n     return the error or best available result.\n5. Return the JSON output block below. Do nothing else.\n\n────────────────────────────────────────\nGRAPH CONT

In [13]:
result = sparql_agent.respond_to("What information is there on engine regulations?")

Calling Tool...


ParseException: Expected SelectQuery, found ')'  (at char 216), (line:1, col:217)

In [11]:
sparql_results = json.loads(result)

print("User Query:", sparql_results["user_query"])
print("SPARQL Query:", sparql_results["sparql_query"])

for res in sparql_results["raw_results"]:
    for part in res:
        print(part)
    print()

User Query: What information is there on engine regulations?
SPARQL Query: SELECT ?s ?p ?o WHERE { ?s ?p ?o . FILTER( regex(str(?s), "engin|motor|regulat|legislat|act", "i") || regex(str(?p), "engin|motor|regulat|legislat|act", "i") || regex(str(?o), "engin|motor|regulat|legislat|act", "i") ) } LIMIT 100


In [12]:
print(sparql_agent.context[-1]["content"])

{
  "user_query": "What information is there on engine regulations?",
  "sparql_query": "SELECT ?s ?p ?o WHERE { ?s ?p ?o . FILTER( regex(str(?s), \"engin|motor|regulat|legislat|act\", \"i\") || regex(str(?p), \"engin|motor|regulat|legislat|act\", \"i\") || regex(str(?o), \"engin|motor|regulat|legislat|act\", \"i\") ) } LIMIT 100",
  "keywords": {
    "extracted": {
      "subject": "regulations",
      "predicate": "",
      "object": ""
    },
    "expanded": {
      "subject": "regulat|legislat|act",
      "predicate": "",
      "object": ""
    }
  },
  "raw_results": [],
  "verbalized_results": [],
  "provenance": [],
  "error": "No results found for the query."
}


#### Agent Instantiation:

In [18]:
receptionist = Legal_Agent(system_prompt=receptionist_prompt, step_prompts=receptionist_step_prompts, api_key=openai.api_key)

legal_expert_lookup = dict()
for expert in legal_expert_list:
    expert_instance = Legal_Agent(
        system_prompt=expert["prompt"],
        step_prompts=expert["step_prompts"],
        tool_table=sparql_batch_tool_dict,
        api_key=expert["api_key"]
        )
    legal_expert_lookup[expert["practice_area"]] = expert_instance

evaluator = Legal_Agent(system_prompt=evaluator_prompt, api_key=openai.api_key)

#### *Receptionist Agent Testing:*

In [80]:
receptionist = Legal_Agent(system_prompt=receptionist_prompt, step_prompts=receptionist_step_prompts, api_key=openai.api_key)

In [62]:
receptionist.reset_context()

In [22]:
receptionist.context

[{'role': 'system',
  'content': '\nYou are a front-facing receptionist that handles inquiries about EU law.\nYou are required to do the following:\n\n1.  Reformulate and improve queries;\n2.  Identify ALL practice areas RELEVANT to the reformulated query;\n3.  Respond to queries accordingly.\n\nYou will be prompted to do each of these tasks, one at a time.\n'},
 {'role': 'user',
  'content': '\n    1.  Reformulate the user\'s query via decomposition and/or adding sufficient context.\n        Ensure that queries/sub-queries are atomic to facilitate downstream SPARQL queries.\n        Collect the resulting query/sub-queries in a list.\n        Determine if the user\'s query is general or complex.\n        If the query is general, proceed to step 3 after the outputting the JSON dictionary.\n\n        Output the following JSON dictionary:\n        {\n            "improved_query": [reformulated_query_list],\n            "next_step": 2 or 3 depending on step 1\n        }\n\n        User que

In [26]:
test_queries = json.loads(receptionist.context[2]["content"])["improved_query"]
test_queries

['What consumer rights does a buyer have under EU and German law for a faulty laptop purchased in Berlin?',
 'What obligations does a retailer have in Germany regarding warranty claims for a product that fails during the warranty period?',
 'What steps should a consumer take when a retailer refuses to honor a warranty for a faulty product in Germany?',
 'What legal remedies are available to a consumer in Germany if a retailer refuses to repair, replace, or refund a faulty laptop under warranty?',
 'What are the time limits for making a warranty claim under German law?',
 'What alternative dispute resolution mechanisms are available for consumer disputes in Germany, such as ADR or small claims court?']

In [81]:
result = receptionist.sequential_reasoning(query="Hi there, I'm Jim. I paid for a laptop at Harvey Norman's in Berlin. It broke down while the warranty was still active. but the store doesn't want to claim responsibility. What should I do? ")

In [85]:
print("Reformulated List of Queries:")
for q in result["improved_query"]:
    print("\t", q)
print()
print("Response:", result["response"])
print("Practice Areas:", result["practice_areas"])

Reformulated List of Queries:
	 What are the consumer rights under EU law for non‑conforming goods purchased in Germany?
	 What obligations does a seller have regarding warranty claims for a laptop bought in Berlin?
	 What steps can a consumer take if the seller refuses to accept responsibility for a warranty claim?
	 What remedies (repair, replacement, refund) are available to a consumer under EU law when a product fails during the warranty period?
	 How can a consumer enforce these rights, including possible legal actions, alternative dispute resolution, or small‑claims procedures in Germany?

Response: We have identified the relevant practice areas for your inquiry (Consumer law, EU law, Commercial law, Litigation/mediation/arbitration, and damage to goods). I will forward your questions to the appropriate experts in these fields so they can provide you with detailed guidance on your rights, the seller’s obligations, possible remedies, and the steps you can take to enforce those rig

In [86]:
result = receptionist.sequential_reasoning("Hi there, what is EU law?")

In [87]:
print("Reformulated List of Queries:")
for q in result["improved_query"]:
    print("\t", q)
print()
print("Response:", result["response"])
print("Practice Areas:", result["practice_areas"])

Reformulated List of Queries:
	 What is the definition of EU law?
	 What are the primary sources of EU law (treaties, regulations, directives, decisions, case law)?
	 What are the main policy areas covered by EU law (e.g., internal market, competition, consumer protection, environment, etc.)?
	 How does EU law interact with the national laws of member states (principle of supremacy, direct effect)?
	 Which EU institutions are responsible for creating, adopting, and interpreting EU law (European Commission, European Parliament, Council of the EU, Court of Justice of the EU)?

Response: EU law is the body of legal rules that apply across the European Union, created by its institutions and binding on both member states and individuals. 

The primary sources of EU law are:
1. **Treaties** – the founding treaties (Treaty on European Union and Treaty on the Functioning of the EU) set out the Union’s objectives and powers.
2. **Regulations** – directly applicable in all member states without 

In [27]:
str(test_queries)

"['What consumer rights does a buyer have under EU and German law for a faulty laptop purchased in Berlin?', 'What obligations does a retailer have in Germany regarding warranty claims for a product that fails during the warranty period?', 'What steps should a consumer take when a retailer refuses to honor a warranty for a faulty product in Germany?', 'What legal remedies are available to a consumer in Germany if a retailer refuses to repair, replace, or refund a faulty laptop under warranty?', 'What are the time limits for making a warranty claim under German law?', 'What alternative dispute resolution mechanisms are available for consumer disputes in Germany, such as ADR or small claims court?']"

#### *Legal Expert Agent Testing:*

In [29]:
test_expert = legal_expert_lookup["Consumer law"]

In [31]:
test_expert.reset_context()

In [30]:
test_expert.context

[{'role': 'system',
  'content': '\n    You are a EU legal researcher in the following practice area: Consumer law.\n    You are required to analyze the given queries/sub-queries STRICTLY WITHIN your practice area.\n    You will be tasked to do the following:\n\n    1.  For each query/sub-query, call a SPARQL sub-agent via the sparql_agent tool.\n    2.  Use the IRAC (Issue-Rules-Application-Conclusion) framework to analyze the results.\n    3.  Organize the results into a coherent argument.\n\n    You will be prompted to do each of these tasks, one at a time.\n    '},
 {'role': 'user',
  'content': '\n        1.  For each query/sub-query, call a SPARQL sub-agent via the sparql_agent tool.\n            ONLY SEARCH FOR INFORMATION WITHIN YOUR PRACTICE AREA.\n\n            Output the following JSON dictionary:\n            {\n                "compiled_results": [list of passed statements/propositions],\n                "queries_to_recall": [keep this blank, it\'s for Step 2],\n          

In [32]:
legal_expert_lookup["Consumer law"].sequential_reasoning(str(test_queries))

Step 1


{'path': 'tools/sparql_agent.py', 'line_start': 1, 'line_end': 400}

#### *Evaluator Agent Testing:*

### **Complete Workflow**

In [23]:
def analyze_query(query):
    # Screening by the Receptionist Agent
    screened_results = receptionist.sequential_reasoning(query)

    print("Finished screening query")

    receptionist_response = screened_results.get("response")
    improved_queries = screened_results.get("improved_queries")
    relev_areas = screened_results.get("practice_areas")

    print("Finished parsing receptionist's response")

    # If the query is simple:
    if not relev_areas:
        print("Proceeding to general response")
        return receptionist_response

    # Otherwise, route the reformulated query(ies) to the relevant Legal Experts:
    else:
        # Route queries to relevant Legal Expert Agents
        print("Calling experts:")

        called_experts = [legal_expert_lookup[area] for area in relev_areas]

        expert_analyses = list()

        def expert_thread(expert_agent):
            print(expert_agent.context[-1])
            return expert_agent.sequential_reasoning(improved_queries)
        
        print("Starting batch calls:")

        with ThreadPoolExecutor() as executor:
            expert_analyses = list(executor.map(expert_thread, called_experts))

        # Output format of a Legal Expert Agent
        #   {
        #       "practice_area": {area},
        #       "supporting_propositions": [list of supporting statements/propositions],
        #       "refuting_propositions": [list of refuting statements/propositions]
        #   }

        formatted_analyses = list()
        for analysis in expert_analyses:
            area = analysis.get("practice_area")
            support_props = analysis.get("supporting_propositions")
            refute_props = analysis.get("refuting_propositions")

            support_props_indexed = [f"{idx + 1}.\t{prop}" for idx, prop in enumerate(support_props)]
            refute_props_indexed = [f"{idx + 1}.\t{prop}" for idx, prop in enumerate(refute_props)]

            support_props_concat = "\n".join(support_props_indexed)
            refute_props_concat = "\n".join(refute_props_indexed)

            pretty_analysis = f"""
            Practice Area:\t{area}

            Supporting Propositions:
            {support_props_concat}

            Refuting Propositions:
            {refute_props_concat}
            -------------------------
            """

            formatted_analyses.append(pretty_analysis)

        concatenated_analyses = "\n".join(formatted_analyses)

        print("Passing analyses to evaluator:")

        final_result = json.loads(evaluator.respond_to(concatenated_analyses))

        # Output format of the Evaluator Agent
        #   {
        #       "evaluation": "a comprehensive evaluation",
        #       "supporting_props": [list of supporting statements/propositions],
        #       "refuting_props": [list of refuting statements/propositions],
        #       "inconsistencies": [list of conflicting/inconsistent rules],
        #       "recs_and strats": [list of recommendations and strategies],
        #       "likely_outcomes": [most likely outcomes, if applicable]
        #   }
        
        final_eval = final_result.get("evaluation")
        supporting_props = final_result.get("supporting_props")
        refuting_props = final_result.get("refuting_props")
        inconsistencies = final_result.get("inconsistencies")
        recs_and_strats = final_result.get("recs_and_strats")
        likely_outcomes = final_result.get("likely_outcomes")

        supporting_props_indexed = [f"{idx + 1}.\t{prop}" for idx, prop in enumerate(supporting_props)]
        refuting_props_indexed = [f"{idx + 1}.\t{prop}" for idx, prop in enumerate(refuting_props)]
        inconsistencies_indexed = [f"{idx + 1}.\t{inconsistency}" for idx, inconsistency in enumerate(inconsistencies)]
        recs_and_strats_indexed = [f"{idx + 1}.\t{rec}" for idx, rec in enumerate(recs_and_strats)]
        likely_outcomes_indexed = [f"{idx + 1}.\t{outcome}" for idx, outcome in enumerate(likely_outcomes)]

        supporting_props_concat = "\n".join(supporting_props_indexed)
        refuting_props_concat = "\n".join(refuting_props_indexed)
        inconsistencies_concat = "\n".join(inconsistencies_indexed)
        recs_and_strats_concat = "\n".join(recs_and_strats_indexed)
        likely_outcomes_concat = "\n".join(likely_outcomes_indexed)

        final_response = f"""
        Evaluation:
        {final_eval}

        Supporting Propositions:
        {supporting_props_concat}

        Refuting Propositions:
        {refuting_props_concat}

        Inconsistencies/Conflicting Rules:
        {inconsistencies_concat}

        Recommendations and Strategies:
        {recs_and_strats_concat}
        """

        if likely_outcomes_concat:
            final_response += "\n"
            final_response += f"""
            Likely Outcomes:
            {likely_outcomes_concat}
            """

        return final_response

In [10]:
query = """
Hi there, I'm Jim.
I paid for a laptop at Harvey Norman's in Berlin.
It broke down while the warranty was still active, but the store doesn't want to claim responsibility.
What should I do?
"""

analyze_query(query)

Finished screening query
Finished parsing receptionist's response
Proceeding to general response


'Thank you for your detailed questions regarding your warranty dispute. I will forward your query to the relevant experts in Commercial law, Consumer law, EU law, Litigation, mediation, arbitration, and damage to goods. They will review your situation and provide you with comprehensive guidance on your rights, obligations, remedies, and the steps you can take.'

In [24]:
complex_query = "How do the regulatory objectives, implementation mechanisms, and legal effects" \
                "differ between early Common Agricultural Policy instruments—specifically Regulation No 18/63 and Regulation No 19/65" \
                "—and later harmonization measures such as Directive 66/402/EEC, "\
                "when accounting for interpretative clarifications introduced via corrigenda "\
                "(e.g., Corrigendum to 1963 measure) and implementing decisions like Decision of 30 April 1962? "\
                "In particular, how do these instruments collectively shape market organization, "\
                "seed certification standards, and Member State obligations, "\
                "and what tensions or complementarities emerge between directly applicable regulations and transposition-dependent directives?"

In [25]:
receptionist.context

[{'role': 'system',
  'content': '\nYou are a front-facing receptionist that handles inquiries about EU law.\nYou are required to do the following:\n\n1.  Reformulate and improve queries;\n2.  Identify ALL practice areas RELEVANT to the reformulated query;\n3.  Respond to queries accordingly.\n\nYou will be prompted to do each of these tasks, one at a time.\n'}]

In [ ]:
receptionist.reset_working_memory()
for expert in legal_expert_lookup.values():
    expert.reset_working_memory()

analyze_query(complex_query)

Finished screening query
Finished parsing receptionist's response
Calling experts:
Starting batch calls:
{'role': 'system', 'content': '\n    You are a EU legal researcher in the following practice area: EU law.\n    You are required to analyze the given queries/sub-queries STRICTLY WITHIN your practice area.\n    You will be tasked to do the following:\n\n    1.  For each query/sub-query, call a SPARQL sub-agent via the call_sparql_subagents tool.\n    2.  Use the IRAC (Issue-Rules-Application-Conclusion) framework to analyze the results.\n    3.  Organize the results into a coherent argument.\n\n    You will be prompted to do each of these tasks, one at a time.\n    '}
{'role': 'system', 'content': '\n    You are a EU legal researcher in the following practice area: Public law.\n    You are required to analyze the given queries/sub-queries STRICTLY WITHIN your practice area.\n    You will be tasked to do the following:\n\n    1.  For each query/sub-query, call a SPARQL sub-agent via 

In [8]:
legal_expert_lookup["Human rights and civil liberties"].context

NameError: name 'legal_expert_lookup' is not defined

In [9]:
json.loads(receptionist.context[-1]["content"]).get("practice_areas")

NameError: name 'receptionist' is not defined